# Policy Gradient Metotları Teorisi

Bu notebook policy gradient ailesini teorik olarak anlatır ve Orbit Wars kod tabanındaki PPO uygulamasıyla ilişkilendirir.

Odak noktaları:
- Değer tabanlı yöntemler ve politika tabanlı yöntemler arasındaki fark
- Log-derivative trick
- REINFORCE
- Baseline, advantage ve actor-critic
- GAE
- PPO clipped objective
- Orbit Wars'taki faktörize politika dağılımı

## 0. Sembol Sözlüğü ve Akış

| Sembol | Anlam |
| --- | --- |
| $\pi_\theta(a\mid s)$ | Actor/politika; durumda aksiyon olasılığı üretir |
| $\tau$ | Trajectory: durum, aksiyon ve ödül dizisi |
| $G_t$ | $t$ anından sonraki iskontolu return |
| $V_\phi(s)$ | Critic/value baseline; durumun beklenen değeri |
| $\hat{A}_t$ | Advantage tahmini; aksiyonun beklenenden iyi/kötü olma sinyali |
| $\delta_t$ | TD residual; GAE ve actor-critic'te advantage yapı taşı |
| $\theta_{old}$ | Rollout'u toplayan eski politika parametreleri |
| $r_t(\theta)$ | PPO importance ratio: yeni politika olasılığı / eski politika olasılığı |

Policy gradient akışı genelde şu zincirdir: rollout topla, return veya advantage hesapla, seçilmiş aksiyonların log olasılığını advantage ile ağırlıklandır, sonra policy/value/entropy kayıplarını birlikte optimize et.

## 1. Policy Gradient Neyi Doğrudan Öğrenir?

Bu notebooktaki REINFORCE, Actor-Critic, A2C ve PPO çizgisi pratikte on-policy kabul edilir. Yani rollout verisi, güncel politikadan veya ona çok yakın eski bir sürümden gelmelidir. Eski veriyi DQN gibi uzun süre replay etmek genellikle doğru değildir; çünkü policy gradient, seçilen aksiyonların olasılıklarını veriyi üreten politika altında yorumlar.

DQN gibi değer tabanlı yöntemler önce $Q(s,a)$ öğrenir, sonra greedy politika çıkarır:

$$\pi(s)=\arg\max_a Q(s,a)$$

Policy gradient yöntemleri ise politikayı doğrudan parametrize eder:

$$\pi_\theta(a\mid s)$$

Amaç fonksiyonu beklenen toplam ödüldür:

$$J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}\left[\sum_{t=0}^{T-1}\gamma^t r_{t+1}\right]$$

Burada $\tau=(s_0,a_0,r_1,s_1,...)$ politikanın ürettiği trajectory'dir.

## 2. Log-Derivative Trick

Policy gradient'in merkezi kimliği:

$$\nabla_\theta p_\theta(x)=p_\theta(x)\nabla_\theta \log p_\theta(x)$$

Bu sayede beklenen değerin gradyanı örneklem üzerinden yazılabilir:

$$\nabla_\theta J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}\left[\nabla_\theta \log P_\theta(\tau)G(\tau)\right]$$

Ortam dinamiği $P(s'\mid s,a)$ parametreye bağlı olmadığı için:

$$\nabla_\theta \log P_\theta(\tau)=\sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)$$

Dolayısıyla temel policy gradient:

$$\nabla_\theta J(\theta)=\mathbb{E}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t\mid s_t)G_t\right]$$

## 3. REINFORCE

REINFORCE Monte Carlo return kullanır:

$$G_t=\sum_{k=t}^{T-1}\gamma^{k-t}r_{k+1}$$

Kayıp minimizasyon formatında:

$$L_{PG}(\theta)=-\frac{1}{B}\sum_t \log\pi_\theta(a_t\mid s_t)G_t$$

Sezgi:

- $G_t$ pozitifse seçilen aksiyonun log olasılığı artırılır.
- $G_t$ negatifse log olasılığı düşürülür.
- Return büyüklüğü güncelleme şiddetini belirler.

REINFORCE unbiased ama yüksek varyanslıdır. Uzun episode, seyrek ödül ve self-play durumlarında bu varyans ciddi sorun olur.

In [ ]:
import torch

def discounted_returns(rewards, dones, gamma=0.99):
    """Episode sonlarından resetlenen Monte Carlo return hesabı."""
    returns = torch.zeros_like(rewards)
    running = 0.0
    for t in reversed(range(len(rewards))):
        running = rewards[t] + gamma * running * (1.0 - dones[t])
        returns[t] = running
    return returns

rewards = torch.tensor([0.0, 0.0, 1.0, 0.0, -1.0])
dones = torch.tensor([0.0, 0.0, 1.0, 0.0, 1.0])
discounted_returns(rewards, dones)

## 3.1. Minimal Stokastik Policy Network

Policy gradient'de ağ doğrudan aksiyon değerleri değil, aksiyon dağılımı üretir. Ayrık aksiyon uzayında bunun pratik karşılığı `Categorical(logits=...)` dağılımıdır.

Aşağıdaki ağ REINFORCE loss'unun neural network üzerinde nasıl kurulduğunu gösterir.

In [ ]:
try:
    import torch
    import torch.nn as nn
    from torch.distributions import Categorical
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Bu hücre için PyTorch gerekli. Jupyter'da orbit_wars_env kernelini seçin.") from exc

class StochasticPolicyNet(nn.Module):
    """Policy gradient ağı: state -> action logits -> Categorical policy."""
    def __init__(self, obs_dim=10, hidden_dim=64, n_actions=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, n_actions),
        )

    def distribution(self, obs):
        logits = self.net(obs)
        return Categorical(logits=logits)

policy = StochasticPolicyNet()
obs = torch.randn(5, 10)
dist = policy.distribution(obs)
actions = dist.sample()
log_probs = dist.log_prob(actions)
returns = torch.tensor([1.0, 0.5, -0.2, 1.3, 0.0])

# REINFORCE: iyi return alan aksiyonların log olasılığı artar.
policy_loss = -(log_probs * returns).mean()

print("actions:", actions)
print("log_probs:", log_probs)
print("policy_loss:", float(policy_loss.detach()))

## 3.2. REINFORCE Update Algoritması

Policy network update'i doğrudan log olasılık üzerinden yapılır. Return veya advantage pozitifse seçilen aksiyonun olasılığı artar; negatifse azalır.

In [ ]:
def reinforce_update(policy, optimizer, obs, actions, returns):
    dist = policy.distribution(obs)
    log_probs = dist.log_prob(actions)
    loss = -(log_probs * returns).mean()

    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    return {
        "loss": float(loss.detach()),
        "entropy": float(dist.entropy().mean().detach()),
        "avg_log_prob": float(log_probs.mean().detach()),
    }

policy = StochasticPolicyNet(obs_dim=10, n_actions=4)
optimizer = torch.optim.Adam(policy.parameters(), lr=1e-3)
obs = torch.randn(8, 10)
with torch.no_grad():
    actions = policy.distribution(obs).sample()
returns = torch.randn(8)
reinforce_update(policy, optimizer, obs, actions, returns)

## 4. Baseline ve Advantage

Policy gradient'de return'den sadece duruma bağlı bir baseline çıkarmak gradyanı bias'lamaz:

$$\nabla_\theta J(\theta)=\mathbb{E}\left[\nabla_\theta\log\pi_\theta(a_t\mid s_t)(G_t-b(s_t))\right]$$

En yaygın baseline value fonksiyonudur:

$$b(s_t)=V_\phi(s_t)$$

Bu durumda kullanılan sinyal advantage olur:

$$A_t=G_t-V_\phi(s_t)$$

Sezgi:

- Bir aksiyon beklenenden iyiyse $A_t>0$, olasılığı artırılır.
- Beklenenden kötüyse $A_t<0$, olasılığı azaltılır.
- Sadece yüksek ödül değil, aynı durum için beklenenden yüksek ödül önemlidir.

## 5. Actor-Critic

Actor-critic iki bileşen kullanır:

- Actor: $\pi_\theta(a\mid s)$
- Critic: $V_\phi(s)$ veya $Q_\phi(s,a)$

Critic TD hedefiyle eğitilebilir:

$$y_t=r_{t+1}+\gamma(1-d_t)V_\phi(s_{t+1})$$

$$L_V(\phi)=\frac{1}{2}(V_\phi(s_t)-y_t)^2$$

Actor ise advantage ile güncellenir:

$$L_\pi(\theta)=-\log\pi_\theta(a_t\mid s_t)\hat{A}_t$$

Critic bias ekleyebilir ama varyansı ciddi azaltır. Pratikte PPO, A2C ve A3C gibi algoritmalar bu aileye girer.

## 6. Generalized Advantage Estimation (GAE)

GAE, bias-varyans dengesini $\lambda$ ile kontrol eder. Önce TD residual:

$$\delta_t=r_{t+1}+\gamma(1-d_t)V(s_{t+1})-V(s_t)$$

Sonra advantage tahmini:

$$\hat{A}_t=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}$$

- $\lambda=0$: tek adımlı TD, düşük varyans, yüksek bias
- $\lambda=1$: Monte Carlo'ya yakın, düşük bias, yüksek varyans
- Pratikte $0.9$ ile $0.98$ arası sık kullanılır

Return genellikle şu şekilde kurulur:

$$R_t = \hat{A}_t + V(s_t)$$

In [ ]:
def compute_gae(rewards, dones, values, next_values, gamma=0.99, lam=0.95):
    """GAE advantage ve value target hesabı."""
    advantages = torch.zeros_like(rewards)
    gae = 0.0
    for t in reversed(range(len(rewards))):
        delta = rewards[t] + gamma * next_values[t] * (1.0 - dones[t]) - values[t]
        gae = delta + gamma * lam * (1.0 - dones[t]) * gae
        advantages[t] = gae
    returns = advantages + values
    return advantages, returns

rewards = torch.tensor([0.0, 0.2, 1.0])
dones = torch.tensor([0.0, 0.0, 1.0])
values = torch.tensor([0.1, 0.2, 0.4])
next_values = torch.tensor([0.2, 0.4, 0.0])
compute_gae(rewards, dones, values, next_values)

## 7. PPO: Büyük Politika Adımlarını Sınırlamak

Vanilla policy gradient, tek update'te politikayı çok fazla değiştirebilir. PPO eski ve yeni politika olasılık oranını kullanır:

$$r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)}$$

Clipped surrogate objective:

$$L^{CLIP}(\theta)=\mathbb{E}\left[\min\left(r_t(\theta)\hat{A}_t, clip(r_t(\theta),1-\epsilon,1+\epsilon)\hat{A}_t\right)\right]$$

Maksimizasyon yerine kodda negatif işaretle minimize edilir:

$$L_{policy}=-L^{CLIP}$$

Bu clipping, avantaj pozitifken olasılığı aşırı artırmayı, avantaj negatifken olasılığı aşırı azaltmayı sınırlar.

In [ ]:
def ppo_clipped_policy_loss(new_logp, old_logp, advantages, clip_range=0.2):
    ratio = torch.exp(new_logp - old_logp)
    unclipped = ratio * advantages
    clipped = torch.clamp(ratio, 1.0 - clip_range, 1.0 + clip_range) * advantages
    return -torch.min(unclipped, clipped).mean()

new_logp = torch.tensor([-0.8, -1.3, -0.2])
old_logp = torch.tensor([-1.0, -1.0, -0.3])
advantages = torch.tensor([1.0, -0.5, 0.2])
ppo_clipped_policy_loss(new_logp, old_logp, advantages)

## 8. Entropy Bonus

Politika çok erken deterministik hale gelirse keşif biter. Entropy bonus dağılımı daha yayvan tutar:

$$L = L_{policy} + c_v L_{value} - c_e H(\pi_\theta(\cdot\mid s))$$

Entropy yüksekse belirsizlik/keşif yüksektir. `entropy_coef` çok yüksek olursa ajan iyi aksiyonlara keskinleşemez; çok düşük olursa erken çöker.

Orbit Wars'ta aksiyon üç parçaya ayrıldığı için entropy de kaynak, hedef ve gemi oranı dağılımlarından gelir. Bir branch'in çökmesi toplam davranışı ciddi etkileyebilir.

## 9. Orbit Wars'taki Faktörize Politika

`model.py::OrbitWarsPolicy` politikayı şu şekilde faktörize eder:

$$\pi(a\mid s)=\pi_{src}(src\mid s)\,\pi_{tgt}(tgt\mid s,src)\,\pi_{ship}(ship\mid s,src,tgt)$$

Log olasılık toplam olur:

$$\log\pi(a\mid s)=\log\pi_{src}+\log\pi_{tgt}+\log\pi_{ship}$$

Bu, policy gradient için kritik bir kolaylıktır. Joint aksiyonun olasılığını tek devasa kategorik dağılım olarak hesaplamak yerine branch log-prob'ları toplanır.

Maskeler şunları sağlar:

- Sahip olunmayan veya geçersiz kaynak seçilmez.
- Padding gezegenleri hedef olamaz.
- NO-OP için hedef/gemi branch'i davranışa dahil edilmez.

`rl/ppo.py::ppo_update`, eski log-prob ile yeni log-prob oranını bu faktörize dağılım üzerinden hesaplar.

## 10. PPO Kayıp Bileşenleri

Orbit Wars PPO uygulamasındaki toplam kayıp:

$$L = L_{policy} + c_v L_{value} - c_e H$$

Kod karşılıkları:

- `policy_loss`: clipped surrogate'ın negatifi
- `value_loss`: value head'in return hedeflerine uzaklığı
- `entropy`: keşif bonusu
- `approx_kl`: eski ve yeni politika arasındaki yaklaşık uzaklık
- `clip_fraction`: oranların ne kadarının clipping sınırını aştığı

`clip_fraction` çok yüksekse update adımı agresif olabilir. `approx_kl` hızla büyüyorsa politika eski veriye göre fazla kaymıştır. Entropy çok hızlı düşüyorsa ajan erken deterministikleşiyor olabilir.

## 11. DQN ile Policy Gradient Karşılaştırması

| Özellik | DQN | Policy Gradient / PPO |
| --- | --- | --- |
| Öğrenilen şey | $Q(s,a)$ | $\pi(a\mid s)$ ve genelde $V(s)$ |
| Veri kullanımı | Off-policy replay | Çoğunlukla on-policy rollout |
| Aksiyon tipi | Ayrık aksiyonlarda doğal | Ayrık ve sürekli aksiyonlarda doğal |
| Keşif | Epsilon-greedy | Stokastik politika + entropy |
| Stabilite aracı | Target network, replay, Double DQN | Clipping, KL takibi, advantage normalize |
| Orbit Wars etkisi | Faktörize Q branch'leri gerekir | Faktörize log-prob doğal çalışır |

Orbit Wars gibi self-play ve maskeli aksiyon uzayı olan oyunlarda PPO daha doğrudan bir politika modeli sunar. DQN ise replay sayesinde sample efficient olabilir ama hareketli self-play dağılımı ve büyük aksiyon uzayı nedeniyle daha dikkatli ayar ister.

## 12. Okuma Haritası

- `model.py`: policy/value ağının Set Transformer encoder'ı ve aksiyon dağılımı
- `rl/rollout.py`: rollout toplama ve advantage/return hazırlığı
- `rl/buffer.py`: PPO minibatch yapısı
- `rl/ppo.py`: clipped policy loss, value loss, entropy ve KL metrikleri
- `rl/train.py`: self-play PPO eğitim döngüsü

Policy gradient eğitiminde tek başına episode reward yeterli değildir. Advantage dağılımı, entropy, approx KL, clip fraction, value loss ve win-rate birlikte değerlendirilmelidir.